In [ ]:
# 1. INSTALACIÓN DE LIBRERÍAS ADICIONALES
# Instalamos pypdf para la extracción de texto desde archivos PDF binarios
!pip install pypdf

import io
import re
import unicodedata
from google.colab import files
from pypdf import PdfReader
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# 2. DESCARGAS OBLIGATORIAS DE NLTK
# Mantenemos las descargas del laboratorio para el procesamiento de texto
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

# 3. INTERFAZ DE CARGA DE ARCHIVO
print("Por favor, sube tu archivo PDF a continuación:")
archivos_subidos = files.upload()

# Obtener el nombre del archivo cargado dinámicamente
nombre_archivo = list(archivos_subidos.keys())[0]
print(f"Archivo cargado con éxito: {nombre_archivo}\n")

# 4. EXTRACCIÓN DE TEXTO DESDE EL PDF
texto_crudo = ""
try:
    # Leemos el binario guardado en la memoria de la sesión
    lector_pdf = PdfReader(nombre_archivo)
    lista_paginas = []

    # Recorremos cada página del documento extrayendo su contenido
    for numero_pagina in range(len(lector_pdf.pages)):
        pagina = lector_pdf.pages[numero_pagina]
        texto_pagina = pagina.extract_text()
        if texto_pagina:
            lista_paginas.append(texto_pagina)

    # Unificamos todo el contenido en una sola cadena de texto gigante
    texto_crudo = " ".join(lista_paginas)
    print("--- FASE 1: EXTRACCIÓN COMPLETADA ---")
    print(f"Páginas procesadas: {len(lector_pdf.pages)}")
    print(f"Caracteres totales extraídos: {len(texto_crudo)}\n")
except Exception as e:
    print(f"Error al leer el archivo PDF: {e}")

# 5. PIPELINE DE PLN (Procesamiento idéntico al de tu laboratorio)
if texto_crudo.strip():
    # A. Normalización Unicode (Remover acentos y estandarizar caracteres)
    # Convierte caracteres como 'á' en 'a' y pasa todo a minúsculas
    texto_normalizado = "".join(
        c for c in unicodedata.normalize('NFD', texto_crudo)
        if unicodedata.category(c) != 'Mn'
    ).lower()

    # B. Expresiones Regulares (Eliminar números y signos de puntuación)
    # Conservamos únicamente letras minúsculas (a-z) y espacios en blanco
    texto_limpio = re.sub(r'[^a-z\s]', '', texto_normalizado)

    # C. Tokenización (Segmentación de palabras independientes)
    tokens = word_tokenize(texto_limpio)

    # D. Remoción de Stopwords (Filtro de palabras vacías en español)
    palabras_vacias = set(stopwords.words('spanish'))

    # Filtramos las stopwords y removemos caracteres sueltos o basura residual
    tokens_finales = [
        token for token in tokens
        if token not in palabras_vacias and len(token) > 1
    ]

    # 6. VISUALIZACIÓN DE LOS RESULTADOS DEL LABORATORIO
    print("--- FASE 2: PIPELINE DE PLN FINALIZADO ---")
    print(f"Muestra del texto limpio (primeros 300 caracteres):\n{texto_limpio[:300]}...\n")
    print(f"Total de tokens extraídos originalmente: {len(tokens)}")
    print(f"Total de tokens finales (sin stopwords): {len(tokens_finales)}\n")

    print("--- MUESTRA DE LOS TOKENS FINALES (Primeros 50) ---")
    print(tokens_finales[:50])
else:
    print("No se pudo ejecutar el pipeline de PLN porque el archivo PDF no contiene texto legible.")


Por favor, sube tu archivo PDF a continuación:


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Saving Practica 5.pdf to Practica 5.pdf
Archivo cargado con éxito: Practica 5.pdf

--- FASE 1: EXTRACCIÓN COMPLETADA ---
Páginas procesadas: 3
Caracteres totales extraídos: 2121

--- FASE 2: PIPELINE DE PLN FINALIZADO ---
Muestra del texto limpio (primeros 300 caracteres):
universidad nacional de ingenieria
facultad de ciencias
escuela profesional de ciencia de la computacion
introduccion a la programacion
practica 
bic

 crear un programa que de al usuario la oportunidad de adivinar un
numero del  al  prefijado en el programa en un maximo de 
intentos en cada pasada ...

Total de tokens extraídos originalmente: 301
Total de tokens finales (sin stopwords): 151

--- MUESTRA DE LOS TOKENS FINALES (Primeros 50) ---
['universidad', 'nacional', 'ingenieria', 'facultad', 'ciencias', 'escuela', 'profesional', 'ciencia', 'computacion', 'introduccion', 'programacion', 'practica', 'bic', 'crear', 'programa', 'usuario', 'oportunidad', 'adivinar', 'numero', 'prefijado', 'programa', 'maximo', 'inte

In [ ]:
# 1. INSTALACIÓN DE LIBRERÍAS Y COMPONENTES DE IDIOMA
# Instalamos el lector de PDF y descargamos el modelo en español de spaCy
!pip install pypdf
!python -m spacy download es_core_news_sm

import io
import re
import unicodedata
import pandas as pd
import spacy
from google.colab import files
from pypdf import PdfReader
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# 2. DESCARGAS OBLIGATORIAS DE NLTK
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

# 3. TABLA DE TRADUCCIÓN DE ESTÁNDAR ANCORA (De tu código original)
TRADUCCION_ANCORA = {
    "categorias": {"v": "Verbo", "n": "Sustantivo", "a": "Adjetivo", "p": "Pronombre", "d": "Determinante", "r": "Adverbio", "s": "Preposición", "c": "Conjunción"},
    "tipo_verbo": {"m": "Principal", "a": "Auxiliar", "s": "Semiauxiliar"},
    "modo":       {"i": "Indicativo", "s": "Subjuntivo", "m": "Imperativo", "n": "Infinitivo", "g": "Gerundio", "p": "Participio"},
    "tiempo":     {"p": "Presente", "i": "Imperfecto", "f": "Futuro", "s": "Pasado", "c": "Condicional"},
    "persona":    {"1": "1ª pers.", "2": "2ª pers.", "3": "3ª pers."},
    "genero":     {"m": "Masculino", "f": "Femenino", "c": "Común"},
    "numero":     {"s": "Singular", "p": "Plural", "n": "Invariable"},
    "tipo_sust":  {"c": "Común", "p": "Propio"},
    "tipo_adj":   {"q": "Calificativo", "o": "Ordinal"},
    "caso":       {"n": "Nominativo", "a": "Acusativo", "d": "Dativo", "o": "Oblicuo"}
}

def mi_pos_ancora(token):
    """Traduce las etiquetas complejas del corpus AnCora a texto legible."""
    tag = token.tag_
    if not tag or len(tag) < 3:
        return "Otro"

    inicial = tag[0]
    cat = TRADUCCION_ANCORA["categorias"].get(inicial, "Otro")

    if inicial == "v" and len(tag) >= 6:
        tipo = TRADUCCION_ANCORA["tipo_verbo"].get(tag[1], "")
        modo = TRADUCCION_ANCORA["modo"].get(tag[2], "")
        if tag[2] in ["i", "s", "m"]:
            tiempo = TRADUCCION_ANCORA["tiempo"].get(tag[3], "")
            pers = TRADUCCION_ANCORA["persona"].get(tag[4], "")
            num = TRADUCCION_ANCORA["numero"].get(tag[5], "")
            return f"{cat} {tipo} ({modo} {tiempo}, {pers} {num})"
        return f"{cat} {tipo} ({modo})"

    elif inicial == "n" and len(tag) >= 5:
        tipo = TRADUCCION_ANCORA["tipo_sust"].get(tag[1], "")
        gen = TRADUCCION_ANCORA["genero"].get(tag[2], "")
        num = TRADUCCION_ANCORA["numero"].get(tag[3], "")
        return f"{cat} {tipo} ({gen} {num})"

    elif inicial == "a" and len(tag) >= 5:
        tipo = TRADUCCION_ANCORA["tipo_adj"].get(tag[1], "")
        gen = TRADUCCION_ANCORA["genero"].get(tag[3], "")
        num = TRADUCCION_ANCORA["numero"].get(tag[4], "")
        return f"{cat} {tipo} ({gen} {num})"

    elif tag.startswith("pp") and len(tag) >= 6:
        pers = TRADUCCION_ANCORA["persona"].get(tag[2], "")
        gen = TRADUCCION_ANCORA["genero"].get(tag[3], "")
        num = TRADUCCION_ANCORA["numero"].get(tag[4], "")
        caso = TRADUCCION_ANCORA["caso"].get(tag[5], "No especificado")
        return f"Pronombre Personal ({pers}, {gen} {num}, Caso: {caso})"

    return cat

# 4. INTERFAZ DE CARGA DE ARCHIVO
print("Por favor, sube tu archivo PDF a continuación:")
archivos_subidos = files.upload()
nombre_archivo = list(archivos_subidos.keys())[0]

# 5. EXTRACCIÓN DE TEXTO
texto_crudo = ""
try:
    lector_pdf = PdfReader(nombre_archivo)
    lista_paginas = []
    for numero_pagina in range(len(lector_pdf.pages)):
        pagina = lector_pdf.pages[numero_pagina]
        texto_pagina = pagina.extract_text()
        if texto_pagina:
            lista_paginas.append(texto_pagina)
    texto_crudo = " ".join(lista_paginas)
    print("\n--- EXTRAÍDO CORRECTAMENTE ---")
except Exception as e:
    print(f"Error al leer el archivo PDF: {e}")

# 6. PIPELINE DE PROCESAMIENTO TRADICIONAL
if texto_crudo.strip():
    # Normalización y Limpieza con RegEx
    texto_normalizado = "".join(
        c for c in unicodedata.normalize('NFD', texto_crudo)
        if unicodedata.category(c) != 'Mn'
    ).lower()
    texto_limpio = re.sub(r'[^a-z\s]', '', texto_normalizado)

    # Tokenización y remoción de Stopwords
    tokens = word_tokenize(texto_limpio)
    palabras_vacias = set(stopwords.words('spanish'))
    tokens_finales = [t for t in tokens if t not in palabras_vacias and len(t) > 1]

    # --- EXPANSIÓN 1: TABLA DE DISTRIBUCIÓN DE FRECUENCIAS ---
    print("\n=== TABLA DE DISTRIBUCIÓN DE FRECUENCIAS (TOP 20) ===")
    serie_tokens = pd.Series(tokens_finales)
    df_frecuencias = serie_tokens.value_counts().reset_index()
    df_frecuencias.columns = ['Token / Palabra', 'Frecuencia Absoluta']
    print(df_frecuencias.head(20).to_string(index=False))

    # --- EXPANSIÓN 2: ANÁLISIS MORFOLÓGICO ANCORA (SPACY) ---
    print("\n=== ANÁLISIS MORFOLÓGICO DETALLADO (MUESTRA DE ENTRADA) ===")
    nlp = spacy.load("es_core_news_sm")

    # Tomamos un fragmento representativo del texto original para no saturar la consola
    # Usamos los primeros 300 caracteres del texto original extraído
    fragmento_original = texto_crudo[:300].strip()
    doc_spacy = nlp(fragmento_original)

    # Aplicamos tu List Comprehension directa llamando al traductor
    pos_espanol_detallado = [mi_pos_ancora(token) for token in doc_spacy]

    print(f"{'Palabra':<14} | {'Tag':<8} | {'Categoría Detallada (AnCora)'}")
    print("-" * 65)
    for token, pos in zip(doc_spacy, pos_espanol_detallado):
        # Evitamos imprimir espacios puros en la tabla para mantener el orden visual
        if not token.text.isspace():
            print(f"{token.text:<14} | {token.tag_:<8} | {pos}")

else:
    print("El archivo PDF no contiene texto procesable.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 52.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Por favor, sube tu archivo PDF a continuación:


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Saving 2026_II_NLP_Sesion3.pdf to 2026_II_NLP_Sesion3 (1).pdf

--- EXTRAÍDO CORRECTAMENTE ---

=== TABLA DE DISTRIBUCIÓN DE FRECUENCIAS (TOP 20) ===
Token / Palabra  Frecuencia Absoluta
            and                    7
            the                    5
            nlp                    3
             of                    3
            for                    3
     techniques                    2
         corpus                    2
           task                    2
             si                    2
       sentence                    2
  normalization                    2
   tokenization                    2
    lowercasing                    2
       cleaning                    2
   multilingual                    2
   segmentation                    2
         tokens                    2
            web                    2
       decoders                    2
       problems                    2

=== ANÁLISIS MORFOLÓGICO DETALLADO (MUESTRA DE ENTRADA) ===
Palabra      

In [ ]:
# 1. INSTALACIÓN DE LIBRERÍAS Y MODELOS
!pip install pypdf
!python -m spacy download es_core_news_sm

import io
import re
import unicodedata
import pandas as pd
import spacy
from google.colab import files
from pypdf import PdfReader
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# Descargas iniciales de NLTK
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

# 2. CONFIGURACIÓN DE TRADUCCIÓN ANCORA
TRADUCCION_ANCORA = {
    "categorias": {"v": "Verbo", "n": "Sustantivo", "a": "Adjetivo", "p": "Pronombre", "d": "Determinante", "r": "Adverbio", "s": "Preposición", "c": "Conjunción"},
    "tipo_verbo": {"m": "Principal", "a": "Auxiliar", "s": "Semiauxiliar"},
    "modo":       {"i": "Indicativo", "s": "Subjuntivo", "m": "Imperativo", "n": "Infinitivo", "g": "Gerundio", "p": "Participio"},
    "tiempo":     {"p": "Presente", "i": "Imperfecto", "f": "Futuro", "s": "Pasado", "c": "Condicional"},
    "persona":    {"1": "1ª pers.", "2": "2ª pers.", "3": "3ª pers."},
    "genero":     {"m": "Masculino", "f": "Femenino", "c": "Común"},
    "numero":     {"s": "Singular", "p": "Plural", "n": "Invariable"},
    "tipo_sust":  {"c": "Común", "p": "Propio"},
    "tipo_adj":   {"q": "Calificativo", "o": "Ordinal"}
}

def mi_pos_ancora(token):
    tag = token.tag_
    if not tag or len(tag) < 3:
        return "Otro"
    inicial = tag[0]
    cat = TRADUCCION_ANCORA["categorias"].get(inicial, "Otro")

    if inicial == "v" and len(tag) >= 6:
        tipo = TRADUCCION_ANCORA["tipo_verbo"].get(tag[1], "")
        modo = TRADUCCION_ANCORA["modo"].get(tag[2], "")
        return f"{cat} {tipo} ({modo})"
    elif inicial == "n" and len(tag) >= 5:
        tipo = TRADUCCION_ANCORA["tipo_sust"].get(tag[1], "")
        gen = TRADUCCION_ANCORA["genero"].get(tag[2], "")
        num = TRADUCCION_ANCORA["numero"].get(tag[3], "")
        return f"{cat} {tipo} ({gen} {num})"
    return cat

# 3. CARGA Y EXTRACCIÓN
print("Sube el archivo PDF para el analisis general:")
archivos_subidos = files.upload()
nombre_archivo = list(archivos_subidos.keys())[0]

texto_crudo = ""
try:
    lector_pdf = PdfReader(nombre_archivo)
    lista_paginas = [pagina.extract_text() for pagina in lector_pdf.pages if pagina.extract_text()]
    texto_crudo = " ".join(lista_paginas)
except Exception as e:
    print(f"Error en la lectura del binario: {e}")

# 4. EJECUCIÓN DEL PIPELINE Y CASOS PRÁCTICOS
if texto_crudo.strip():
    # Limpieza tradicional
    texto_normalizado = "".join(c for c in unicodedata.normalize('NFD', texto_crudo) if unicodedata.category(c) != 'Mn').lower()
    texto_limpio = re.sub(r'[^a-z\s]', '', texto_normalizado)

    tokens = word_tokenize(texto_limpio)
    palabras_vacias = set(stopwords.words('spanish'))
    tokens_finales = [t for t in tokens if t not in palabras_vacias and len(t) > 1]

    # CASO PRÁCTICO 1: Tabla de Frecuencias y Exportación Automática
    serie_tokens = pd.Series(tokens_finales)
    df_frecuencias = serie_tokens.value_counts().reset_index()
    df_frecuencias.columns = ['Token', 'Frecuencia']

    # Guardar a CSV local en Colab
    nombre_csv = "frecuencias_vocabulario.csv"
    df_frecuencias.to_csv(nombre_csv, index=False, encoding='utf-8')
    print(f"\nArchivo de frecuencias guardado localmente como: {nombre_csv}")

    print("\n--- TOP 10 PALABRAS MÁS FRECUENTES ---")
    print(df_frecuencias.head(10).to_string(index=False))

    # CASO PRÁCTICO 2: Filtrado Morfológico Selectivo (Sustantivos Propios y Verbos)
    # Útil para responder: ¿Quiénes participan (Sustantivos) y Qué acciones realizan (Verbos)?
    print("\n--- FILTRADO SELECTIVO: VERBOS Y SUSTANTIVOS PROPIOS (Muestra del Texto) ---")
    nlp = spacy.load("es_core_news_sm")
    doc_spacy = nlp(texto_crudo[:500]) # Procesamos un fragmento para control de consola

    print(f"{'Palabra':<15} | {'Tipo AnCora':<35}")
    print("-" * 55)
    for token in doc_spacy:
        # np = Nombre Propio (Sustantivo propio), v = Verbo en el estándar AnCora
        if token.tag_.startswith("np") or token.tag_.startswith("v"):
            analisis = mi_pos_ancora(token)
            print(f"{token.text:<15} | {analisis:<35}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 89.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Sube el archivo PDF para el analisis general:


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Saving lec-5 RequirementsEng..pdf to lec-5 RequirementsEng..pdf

Archivo de frecuencias guardado localmente como: frecuencias_vocabulario.csv

--- TOP 10 PALABRAS MÁS FRECUENTES ---
       Token  Frecuencia
         the         137
          to          61
requirements          48
        user          46
          of          44
          be          41
         and          38
        will          38
          as          30
         req          30

--- FILTRADO SELECTIVO: VERBOS Y SUSTANTIVOS PROPIOS (Muestra del Texto) ---
Palabra         | Tipo AnCora                        
-------------------------------------------------------


In [1]:
import re

# TEXTO SUCIO DE PRUEBA (Caso de estudio: Historial clínico / Reporte legal simulado)
corpus_prueba = """
El usuario Juan Perez (ID: 456-A) solicito soporte el dia 24/11/2025.
Su correo es juan.perez@estudio-legal.com y su telefono es +34 611-223-344.
Costo del trámite: $1,550.45 de urgencia.
Visitar la pagina web oficial https://tramites-legales.es para mas informacion.
Texto con errores de formato continuo......   limpiar espacios!!!
"""

print("=== LABORATORIO DE EXPRESIONES REGULARES PARA NLP ===")
print("Texto original de analisis:\n", corpus_prueba)
print("=" * 60)

# ---------------------------------------------------------------------
# INSTRUCCIÓN 1: Extracción de Entidades Numéricas con Estructura Fija (Fechas)
# Uso: Identificar marcas temporales dentro de documentos.
# ---------------------------------------------------------------------
patron_fecha = r'\b\d{2}/\d{2}/\d{4}\b'
fechas_encontradas = re.findall(patron_fecha, corpus_prueba)
print("\n1. Fechas encontradas (Patron: dd/mm/aaaa):")
print(fechas_encontradas)

# ---------------------------------------------------------------------
# INSTRUCCIÓN 2: Extracción de Cuentas de Correo Electrónico
# Uso: Captura automatizada de datos de contacto o anonimización de datos (Data Masking).
# ---------------------------------------------------------------------
patron_email = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
emails_encontrados = re.findall(patron_email, corpus_prueba)
print("\n2. Correos electronicos detectados:")
print(emails_encontrados)

# ---------------------------------------------------------------------
# INSTRUCCIÓN 3: Extracción de Direcciones URL completas
# Uso: Limpieza de enlaces web que actúan como ruido en tareas de clasificación.
# ---------------------------------------------------------------------
patron_url = r'https?://[a-zA-Z0-9.-]+(?:/[a-zA-Z0-9./_-]*)?'
urls_encontradas = re.findall(patron_url, corpus_prueba)
print("\n3. Enlaces web (URLs) identificados:")
print(urls_encontradas)

# ---------------------------------------------------------------------
# INSTRUCCIÓN 4: Extracción de Valores Monetarios (Cifras Financieras)
# Uso: Aislar importes económicos de multas, contratos o presupuestos.
# ---------------------------------------------------------------------
patron_moneda = r'\$[0-9,]+\.[0-9]{2}'
valores_moneda = re.findall(patron_moneda, corpus_prueba)
print("\n4. Valores financieros/moneda detectados:")
print(valores_moneda)

# ---------------------------------------------------------------------
# INSTRUCCIÓN 5: Segmentación / Limpieza Avanzada de Ruido Textual
# Uso: Corregir problemas de espaciado y caracteres repetidos antes de tokenizar.
# ---------------------------------------------------------------------
# A. Reemplazar multiples puntos seguidos por un solo punto espacio
texto_sin_puntos = re.sub(r'\.{2,}', '. ', corpus_prueba)

# B. Eliminar caracteres que no sean letras, espacios o signos basicos de puntuacion
texto_alfabetico = re.sub(r'[^a-zA-ZáéíóúÁÉÍÓÚñÑ\s]', '', texto_sin_puntos)

# C. Normalizar espacios en blanco duplicados, tabulaciones o saltos de linea
texto_espacios_limpios = re.sub(r'\s+', ' ', texto_alfabetico).strip()

print("\n5. Resultado del texto despues del Pipeline de limpieza con RegEx:")
print(texto_espacios_limpios)

# ---------------------------------------------------------------------
# INSTRUCCIÓN 6: Ejercicio de Anonimización (Caso Práctico de Privacidad)
# Uso: Reemplazar datos sensibles por etiquetas genericas para cumplir leyes de proteccion de datos.
# ---------------------------------------------------------------------
texto_anonimizado = re.sub(patron_email, "[CORREO_ANONIMIZADO]", corpus_prueba)
texto_anonimizado = re.sub(r'\+?\d{2,3}[\s-]?\d{3}[\s-]?\d{3}[\s-]?\d{3}', "[TELEFONO_ANONIMIZADO]", texto_anonimizado)

print("\n6. Documento Anonimizado (Caso Practico de Seguridad):")
print(texto_anonimizado)


=== LABORATORIO DE EXPRESIONES REGULARES PARA NLP ===
Texto original de analisis:
 
El usuario Juan Perez (ID: 456-A) solicito soporte el dia 24/11/2025.
Su correo es juan.perez@estudio-legal.com y su telefono es +34 611-223-344.
Costo del trámite: $1,550.45 de urgencia.
Visitar la pagina web oficial https://tramites-legales.es para mas informacion.
Texto con errores de formato continuo......   limpiar espacios!!!


1. Fechas encontradas (Patron: dd/mm/aaaa):
['24/11/2025']

2. Correos electronicos detectados:
['juan.perez@estudio-legal.com']

3. Enlaces web (URLs) identificados:
['https://tramites-legales.es']

4. Valores financieros/moneda detectados:
['$1,550.45']

5. Resultado del texto despues del Pipeline de limpieza con RegEx:
El usuario Juan Perez ID A solicito soporte el dia Su correo es juanperezestudiolegalcom y su telefono es Costo del trámite de urgencia Visitar la pagina web oficial httpstramiteslegaleses para mas informacion Texto con errores de formato continuo limpiar 

In [2]:
pip install reportlab docling


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.8/303.8 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 100.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3

In [4]:
import os
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors

# Configuración del documento PDF
pdf_filename = "factura_ejercicio.pdf"
doc = SimpleDocTemplate(pdf_filename, pagesize=letter, rightMargin=30, leftMargin=30, topMargin=30, bottomMargin=30)
story = []
styles = getSampleStyleSheet()

# Encabezado de la factura / orden de compra
story.append(Paragraph("<b>ORDEN DE COMPRA: OC-2026-8941</b>", styles["Heading1"]))
story.append(Spacer(1, 10))
story.append(Paragraph("<b>Proveedor:</b> Tecnologia Global S.A.", styles["Normal"]))
story.append(Paragraph("<b>Fecha de Emision:</b> 15/09/2026", styles["Normal"]))
story.append(Spacer(1, 15))

# Estructura tabular de los ítems adquiridos
data = [
    ["Codigo / SKU", "Descripcion del Producto", "Cantidad", "Precio Unitario", "Subtotal"],
    ["SKU-9941", "Monitor Asus 24\" ProArt", "2", "USD 250.00", "USD 500.00"],
    ["SKU-1024", "Laptop Lenovo ThinkPad T14", "1", "USD 1200.00", "USD 1200.00"],
    ["SKU-0089", "Teclado Mecanico Logitech G", "5", "USD 80.00", "USD 400.00"],
]

# Estilizado de la tabla para simular un documento comercial
tabla_productos = Table(data, colWidths=[80, 220, 60, 90, 90])
tabla_productos.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#2c3e50')),
    ('TEXTCOLOR', (0,0), (-1,0), colors.whitesmoke),
    ('ALIGN', (0,0), (-1,-1), 'LEFT'),
    ('BOTTOMPADDING', (0,0), (-1,0), 6),
    ('BACKGROUND', (0,1), (-1,-1), colors.HexColor('#f8f9fa')),
    ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor('#bdc3c7')),
]))
story.append(tabla_productos)
story.append(Spacer(1, 15))

# Bloque final de cierre monetario
story.append(Paragraph("<b>MONTO TOTAL GENERAL: USD 2100.00</b>", styles["Normal"]))

# Compilación física del documento
doc.build(story)
print(f" PDF generado exitosamente en: {os.path.abspath(pdf_filename)}")


 PDF generado exitosamente en: /content/factura_ejercicio.pdf


In [5]:
import re
from docling.document_converter import DocumentConverter

def extraer_datos_factura():
    # 1. Utilizar Docling para procesar el Layout espacial y extraer el texto como Markdown
    print(" Leyendo PDF espacialmente con Docling...")
    converter = DocumentConverter()
    result = converter.convert("factura_ejercicio.pdf")

    # Exportamos a formato Markdown plano (ideal para aplicar Regex de PLN)
    texto_documento = result.document.export_to_markdown()
    print(" Documento convertido a texto estructurado.\n")

    # 2. Definición de Patrones de Expresiones Regulares (Regex)
    patron_oc = r"ORDEN DE COMPRA:\s*(OC-\d{4}-\d{4})"
    patron_proveedor = r"Proveedor:\s*([^\n]+)"
    patron_fecha = r"Fecha de Emision:\s*(\d{2}/\d{2}/\d{4})"
    patron_total = r"MONTO TOTAL GENERAL:\s*USD\s*([\d.]+)"

    # Patrón avanzado para capturar los renglones de la tabla en formato Markdown de Docling
    # Captura: SKU | Descripción | Cantidad | Precio Unitario | Subtotal
    patron_items = r"\|?\s*(SKU-\d+)\s*\|\s*([^\|]+)\|\s*(\d+)\s*\|\s*USD\s*([\d.]+)\s*\|\s*USD\s*([\d.]+)"

    # 3. Extracción de Metadatos Principales (Cabecera)
    match_oc = re.search(patron_oc, texto_documento)
    match_prov = re.search(patron_proveedor, texto_documento)
    match_fecha = re.search(patron_fecha, texto_documento)
    match_total = re.search(patron_total, texto_documento)

    print("=" * 60)
    print(" RESULTADOS DE LA EXTRACCIÓN (PLN + REGEX)")
    print("=" * 60)
    print(f" Código de OC:  {match_oc.group(1) if match_oc else 'No detectado'}")
    print(f" Proveedor:     {match_prov.group(1).strip() if match_prov else 'No detectado'}")
    print(f" Fecha Emisión: {match_fecha.group(1) if match_fecha else 'No detectado'}")
    print(f" Monto Total:   USD {match_total.group(1) if match_total else 'No detectado'}")
    print("-" * 60)

    # 4. Extracción Iterativa del Detalle de la Tabla (Ítems)
    print(" DETALLE DE ÍTEMS DETECTADOS:")
    items_encontrados = re.findall(patron_items, texto_documento)

    for item in items_encontrados:
        sku, descripcion, cantidad, precio_unitario, subtotal = item
        print(f"   [{sku}] {descripcion.strip()} -> Cantidad: {cantidad} | Unitario: USD {precio_unitario} | Subtotal: USD {subtotal}")
    print("=" * 60)

if __name__ == "__main__":
    extraer_datos_factura()


ERROR:docling.datamodel.stage_model_specs:Preset 'granite_vision_v4' already registered for ChartExtractionVlmEngineOptions


 Leyendo PDF espacialmente con Docling...


[INFO] 2026-09-21 01:27:56,147 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-09-21 01:27:56,153 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-09-21 01:27:56,157 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.9.2/torch/PP-OCRv6/det/PP-OCRv6_det_small.pth
[INFO] 2026-09-21 01:27:57,641 [RapidOCR] download_file.py:82: Download size: 9.77MB
[INFO] 2026-09-21 01:27:57,825 [RapidOCR] download_file.py:95: Successfully saved to: /usr/local/lib/python3.13/dist-packages/rapidocr/models/PP-OCRv6_det_small.pth
[INFO] 2026-09-21 01:27:57,829 [RapidOCR] main.py:50: Using /usr/local/lib/python3.13/dist-packages/rapidocr/models/PP-OCRv6_det_small.pth
[INFO] 2026-09-21 01:27:58,125 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-09-21 01:27:58,127 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-09-21 01:27:58,130 [RapidOCR] download_file.py:68: Initiating download: https://www.models

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/conv.py:548: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1024.)
  return F.conv2d(


 Documento convertido a texto estructurado.

 RESULTADOS DE LA EXTRACCIÓN (PLN + REGEX)
 Código de OC:  OC-2026-8941
 Proveedor:     Tecnologia Global S.A.
 Fecha Emisión: 15/09/2026
 Monto Total:   USD 2100.00
------------------------------------------------------------
 DETALLE DE ÍTEMS DETECTADOS:
   [SKU-9941] Monitor Asus 24" ProArt -> Cantidad: 2 | Unitario: USD 250.00 | Subtotal: USD 500.00
   [SKU-1024] Laptop Lenovo ThinkPad T14 -> Cantidad: 1 | Unitario: USD 1200.00 | Subtotal: USD 1200.00
   [SKU-0089] Teclado Mecanico Logitech G -> Cantidad: 5 | Unitario: USD 80.00 | Subtotal: USD 400.00
